In [ ]:
# importing data set
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)

RAW = {
    "kaub_level":   "../data/raw/water_level_15min_kaub.csv",
    "bingen_level": "../data/raw/bingen_water_level15_min.csv",
    "discharge":    "../data/raw/kaub_water_discharge.csv",
}
WEATHER_HOURLY = "../data/processed/kaub_water_weather_hourly.csv"
WEATHER_DAILY  = "../data/processed/kaub_water_weather_daily.csv"

OUT_HOURLY = "../data/processed/kaub_hydro_weather_hourly.csv"
OUT_DAILY  = "../data/processed/kaub_hydro_weather_daily.csv"

In [ ]:
def load_raw(path: str, name: str) -> pd.Series:
    """Load and clean raw 15-minute water-level data from WSV."""

    # Read the raw CSV file using a semicolon as the separator
    d = pd.read_csv(path, sep=";")

    # Convert timestamps to datetime format
    d["timestamp"] = pd.to_datetime(d["timestamp"])

    # Convert water-level values to numeric and replace invalid values with NaN
    d["value"] = pd.to_numeric(d["value"], errors="coerce")

    # Generate a data quality report before removing duplicates
    report = {
        "rows": len(d),
        "duplicate timestamps": int(d["timestamp"].duplicated().sum()),
        "non-numeric values": int(d["value"].isna().sum()),
        "first": d["timestamp"].min(),
        "last": d["timestamp"].max(),
    }

    # Handle duplicate timestamps by calculating their mean water level
    # Sort chronologically and assign the station name to the Series
    s = d.groupby("timestamp")["value"].mean().sort_index().rename(name)

    # Record the number of observations after removing duplicate timestamps
    report["rows after dedup"] = len(s)

    # Display the data quality report for each station
    print(f"--- {name} ---")
    for k, v in report.items():
        print(f"  {k:22s}: {v}")

    # Return the cleaned water-level time series
    return s


# Load and clean raw water-level data for all stations defined in RAW
# Store the cleaned time series in a dictionary using station names as keys
series = {name: load_raw(path, name) for name, path in RAW.items()}

In [ ]:
# Create a dictionary to store duplicate timestamp information for each station
dup_check = {}

# Loop through all stations and their corresponding raw data files
for name, path in RAW.items():

    # Read the raw CSV file using a semicolon as the separator
    d = pd.read_csv(path, sep=";")

    # Convert timestamps to datetime format
    d["timestamp"] = pd.to_datetime(d["timestamp"])

    # Identify all rows with duplicate timestamps, including their first occurrence
    dups = d.loc[d["timestamp"].duplicated(keep=False), "timestamp"]

    # Summarize duplicate timestamp information for each station
    dup_check[name] = {

        # Count the total number of rows involved in duplicate timestamps
        "n duplicated rows": len(dups),

        # Count the number of unique dates affected by duplicate timestamps
        "distinct dates affected": dups.dt.date.nunique(),

        # Identify the months in which duplicate timestamps occur
        "months affected": sorted(dups.dt.month.unique().tolist()),

        # Identify the hours in which duplicate timestamps occur
        "hours affected": sorted(dups.dt.hour.unique().tolist()),

    }

# Convert the duplicate summary into a DataFrame for comparison across stations
pd.DataFrame(dup_check).T

In [ ]:
# Load the processed daily hydro-weather dataset
final_merged_df = pd.read_csv(
    "../data/processed/kaub_hydro_weather_daily.csv"
)

# Identify all numeric columns in the dataset
numeric_cols = final_merged_df.select_dtypes(include="number").columns

# Round numeric values to the nearest integer
# Convert them to nullable integers (Int64) to preserve missing values
final_merged_df[numeric_cols] = (
    final_merged_df[numeric_cols]
    .round()
    .astype("Int64")
)

# Display the first five rows of the updated dataset
final_merged_df.head()

In [ ]:
#final merged data
final_merged_df.head()

In [ ]:
#assigned to df 
df = final_merged_df

In [ ]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Select all numeric columns
#numeric_cols = df.select_dtypes(include="number").columns

# Round values and convert to integers
#df[numeric_cols] = df[numeric_cols].round(0).astype("Int64")

# Sort by date
df = df.sort_values("date").reset_index(drop=True)

# Display the DataFrame
df.head()

In [ ]:
# Count the number of days for each recorded number of available hourly observations
# Sort the results by the number of hours to identify days with missing or extra observations
df["hours_available"].value_counts().sort_index()

In [ ]:
# Select the main hydrological variables for correlation analysis
cols = [
    "water_level_mean",
    "bingen_level_mean",
    "discharge_mean"
]

# Calculate the Pearson correlation coefficients between the selected variables
# Identify the strength and direction of their linear relationships
df[cols].corr()

In [ ]:
# Import libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size for the correlation heatmap
plt.figure(figsize=(7, 5))

# Create a heatmap to visualize correlations between hydrological variables
sns.heatmap(
    df[cols].corr(),  # Calculate Pearson correlation coefficients
    annot=True,       # Display correlation values inside each cell
    fmt=".2f",        # Show correlation values with two decimal places
    cmap="coolwarm",  # Use a color gradient to represent correlation strength
    vmin=-1,          # Set the minimum correlation value
    vmax=1            # Set the maximum correlation value
)

# Add a descriptive title to the heatmap
plt.title("Correlation of Hydrological Variables")

# Display the correlation heatmap
plt.show()

In [ ]:
# Calculate the correlation between Kaub water levels and lagged Bingen water levels
# Test lags from 0 to 7 days to examine the upstream-downstream relationship
for lag in range(0, 8):

    # Shift Bingen water levels by the specified number of days
    # Calculate the Pearson correlation with Kaub water levels
    corr = df["water_level_mean"].corr(
        df["bingen_level_mean"].shift(lag)
    )

    # Display the correlation coefficient for each lag
    print(f"Bingen lag {lag}: r = {corr:.3f}")

In [ ]:
# Calculate the correlation between Kaub water levels and lagged discharge values
# Test lags from 0 to 7 days to examine their temporal relationship
for lag in range(0, 8):

    # Shift discharge values by the specified number of days
    # Calculate the Pearson correlation with Kaub water levels
    corr = df["water_level_mean"].corr(
        df["discharge_mean"].shift(lag)
    )

    # Display the correlation coefficient for each lag
    print(f"Discharge lag {lag}: r = {corr:.3f}")

In [ ]:
# Calculate cumulative precipitation over different time windows
# Rolling sums capture the effect of recent rainfall on river water levels

# Calculate total precipitation over the current and previous 2 days
df["precip_3d"] = df["precipitation_sum"].rolling(3).sum()

# Calculate total precipitation over the current and previous 6 days
df["precip_7d"] = df["precipitation_sum"].rolling(7).sum()

# Calculate total precipitation over the current and previous 13 days
df["precip_14d"] = df["precipitation_sum"].rolling(14).sum()

# Calculate total precipitation over the current and previous 29 days
df["precip_30d"] = df["precipitation_sum"].rolling(30).sum()

In [ ]:
# Calculate the correlation between water levels and cumulative precipitation over different time windows
# Identify which rainfall period has the strongest linear relationship with Kaub water levels

rain_cols = [
    "precipitation_sum",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d"
]

df[rain_cols + ["water_level_mean"]].corr()["water_level_mean"]

In [ ]:
# Visualize the correlations among hydrological and weather variables
# Identify relationships with water levels and potential multicollinearity among features

features = [
    "water_level_mean",
    "temperature_mean",
    "humidity_mean",
    "precipitation_sum",
    "pressure_mean",
    "wind_speed_mean",
    "bingen_level_mean",
    "discharge_mean",
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d"
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    df[features].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Multivariate Correlation Matrix")

plt.show()

In [ ]:
#Fixing the target feature
TARGET = "water_level_mean"

In [ ]:
# Kaub water-level lags
df["level_lag_1"] = df["water_level_mean"].shift(1)
df["level_lag_2"] = df["water_level_mean"].shift(2)
df["level_lag_3"] = df["water_level_mean"].shift(3)
df["level_lag_7"] = df["water_level_mean"].shift(7)
df["level_lag_14"] = df["water_level_mean"].shift(14)

In [ ]:
# Calculate the daily change in Kaub water level compared to the previous day

df["level_change_1d"] = df["water_level_mean"].diff(1)

In [ ]:
# Create lagged Bingen water-level features for the previous 1, 2, and 3 days
# Capture the influence of historical upstream water levels on Kaub

df["bingen_lag_1"] = df["bingen_level_mean"].shift(1)

df["bingen_lag_2"] = df["bingen_level_mean"].shift(2)

df["bingen_lag_3"] = df["bingen_level_mean"].shift(3)

In [ ]:
# Create lagged discharge features for the previous 1, 2, and 3 days
# Capture the influence of historical river discharge on Kaub water levels

df["discharge_lag_1"] = df["discharge_mean"].shift(1)

df["discharge_lag_2"] = df["discharge_mean"].shift(2)

df["discharge_lag_3"] = df["discharge_mean"].shift(3)

In [ ]:
# Calculate the daily change in river discharge compared to the previous day

df["discharge_change_1d"] = df["discharge_mean"].diff(1)

In [ ]:
# Calculate cumulative precipitation over 3, 7, 14, and 30 days
# Capture the influence of recent rainfall on river water levels

df["precip_3d"] = df["precipitation_sum"].rolling(3).sum()

df["precip_7d"] = df["precipitation_sum"].rolling(7).sum()

df["precip_14d"] = df["precipitation_sum"].rolling(14).sum()

df["precip_30d"] = df["precipitation_sum"].rolling(30).sum()

In [ ]:
# Calculate the average temperature over 3 and 7 days
# Capture short-term temperature trends that may influence river water levels

df["temp_3d"] = df["temperature_mean"].rolling(3).mean()

df["temp_7d"] = df["temperature_mean"].rolling(7).mean()

In [ ]:
# Create cyclical day-of-year features using sine and cosine transformations
# Capture seasonal patterns while preserving the continuity between the end and start of each year

import numpy as np

df["day_of_year"] = df["date"].dt.dayofyear

df["doy_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["doy_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

In [ ]:
# Create target variables for forecasting Kaub water levels from 1 to 7 days ahead
# Shift water levels backward to align each future target with the current day's features

for h in range(1, 8):

    df[f"target_day_{h}"] = df["water_level_mean"].shift(-h)

In [ ]:
# Define the input features for multivariate water-level forecasting
# Include historical river conditions, weather patterns, and seasonal variations

FEATURES = [
    # Kaub history
    "level_lag_1",
    "level_lag_2",
    "level_lag_3",
    "level_lag_7",
    "level_lag_14",
    "level_change_1d",

    # Upstream Bingen
    "bingen_lag_1",
    "bingen_lag_2",
    "bingen_lag_3",

    # Discharge
    "discharge_lag_1",
    "discharge_lag_2",
    "discharge_lag_3",
    "discharge_change_1d",

    # Rainfall memory
    "precip_3d",
    "precip_7d",
    "precip_14d",
    "precip_30d",

    # Temperature
    "temp_3d",
    "temp_7d",

    # Seasonality
    "doy_sin",
    "doy_cos"
]

In [ ]:
# Define the target columns for predicting Kaub water levels from 1 to 7 days ahead

Target_cols = [
    f"target_day_{h}"
    for h in range(1, 8)
]

In [ ]:
# Calculate correlations between input features and future water levels for all seven forecast horizons
# Identify how the linear relationship between each feature and the target changes over time

TARGETS = [
    f"target_day_{h}"
    for h in range(1, 8)
]

corr_horizons = df[
    FEATURES + TARGETS
].corr().loc[FEATURES, TARGETS]

display(corr_horizons.round(3))

In [ ]:
# Heatmap : Correlation between engineered features
feature_corr = df[FEATURES].corr()

plt.figure(figsize=(16, 13))

sns.heatmap(
    feature_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title(
    "Correlation Matrix of Multivariate Features"
)

plt.tight_layout()
plt.show()

In [ ]:
model_df = df[
    ["date"] + FEATURES + TARGETS
].dropna().copy()

print(model_df.shape)
print(model_df["date"].min())
print(model_df["date"].max())

In [ ]:
# Print features
print("FEATURES:", FEATURES)

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nMissing features:")
print([col for col in FEATURES if col not in df.columns])

In [ ]:
# Kaub water level lag features
LAGS = [0, 1, 2, 3, 4, 5, 6, 7, 14, 21, 28]

for lag in LAGS:
    df[f"lag_{lag}"] = (
        df["water_level_mean"].shift(lag)
    )

In [ ]:
# Seven-day forecasting targets
TARGETS = []

for h in range(1, 8):

    target_col = f"target_day_{h}"

    df[target_col] = (
        df["water_level_mean"].shift(-h)
    )

    TARGETS.append(target_col)

In [ ]:
# Check whether all required columns exist
missing_cols = [
    col for col in FEATURES + TARGETS
    if col not in df.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)

else:
    # Calculate correlations
    corr_horizons = (
        df[FEATURES + TARGETS]
        .corr()
        .loc[FEATURES, TARGETS]
    )

    display(corr_horizons.round(3))

In [ ]:
plt.figure(figsize=(12, 10))

sns.heatmap(
    corr_horizons,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title(
    "Feature Correlation with Kaub Water Level "
    "Across 7 Forecast Horizons"
)

plt.xlabel("Forecast Horizon")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

In [ ]:
# Prepare the univariate time-series dataset using historical Kaub water levels
# Create lag features and a target variable for forecasting water levels 7 days ahead

HORIZON = 7
H = HORIZON

LAGS = [0, 1, 2, 3, 4, 5, 6, 7, 14, 21, 28]

# Select Kaub daily mean water level
s = df["water_level_mean"]

data = pd.DataFrame(index=s.index)

for L in LAGS:
    data[f"lag_{L}"] = s.shift(L)

data["target"] = s.shift(-H)

FEATURES = [f"lag_{L}" for L in LAGS]

data.head(35)

In [ ]:
# Validate the alignment of lag features and the 7-day-ahead target
# Confirm that the current water level and future target correspond to the correct dates

i = 400

row = data.iloc[i]

assert row["lag_0"] == s.iloc[i]

assert row["target"] == s.iloc[i + H]

print(
    f"On {data.index[i]}: "
    f"today={row['lag_0']:.2f}, "
    f"target(+{H}d)={row['target']:.2f}"
)

In [ ]:
# Check the index type and values to verify the time-series date index

print(type(data.index))

print(data.index)

In [ ]:
# Set the date as a sorted datetime index for time-series analysis
# Select the daily mean Kaub water level as the target series

df["date"] = pd.to_datetime(df["date"])

df = df.set_index("date").sort_index()

s = df["water_level_mean"]

In [ ]:
# Define the forecast horizon and test period, then verify the dataset structure
# Check the date index, input features, data dimensions, and target availability

H = 7

TEST_YEARS = 3

print("Index type:", type(data.index))

print("Features:", FEATURES)

print("Data shape:", data.shape)

print("Target exists:", "target" in data.columns)

print("First date:", data.index.min())

print("Last date:", data.index.max())

In [ ]:
print(df.columns.tolist())
print(df.head(3))

In [ ]:
print(type(df.index))

In [ ]:
df.index = pd.to_datetime(df.index)
df = df.sort_index()

In [ ]:
# Prepare the multivariate dataset with a datetime index and selected input features
# Set the 7-day-ahead water level as the target and verify the dataset structure

import pandas as pd

df.index = pd.to_datetime(df.index)
df = df.sort_index()

FEATURES = [
    "lag_0", "lag_1", "lag_2", "lag_3",
    "lag_7", "lag_14", "lag_28",
    "precip_3d", "precip_7d", "precip_14d", "precip_30d",
    "temp_3d", "temp_7d",
    "bingen_lag_1", "bingen_lag_2", "bingen_lag_3",
    "doy_sin", "doy_cos"
]

data = df[FEATURES + ["target_day_7"]].copy()
data = data.rename(columns={"target_day_7": "target"})

print("df index:", type(df.index))
print("data index:", type(data.index))
print("Last date:", data.index.max())

In [ ]:
# Split the time-series data into training and test sets using the last 3 years for testing
# Apply a 7-day embargo to prevent overlapping forecast targets and verify temporal separation

H = 7

TEST_YEARS = 3

split_date = data.index.max() - pd.DateOffset(years=TEST_YEARS)

train = data.loc[:split_date - pd.Timedelta(days=H)].dropna()

test = data.loc[split_date + pd.Timedelta(days=1):].dropna()

X_train, y_train = train[FEATURES], train["target"]

X_test, y_test = test[FEATURES], test["target"]

print(f"train: {train.index.min().date()} → {train.index.max().date()} n={len(train)}")

print(f"test : {test.index.min().date()} → {test.index.max().date()} n={len(test)}")

assert train.index.max() + pd.Timedelta(days=H) < test.index.min(), "embargo violated"

In [ ]:
# Split the training data into model-fitting and validation sets, using the last 2 years for validation
# Apply a 7-day embargo to prevent target overlap and ensure temporal separation

VALID_YEARS = 2

validation_start = (
    train.index.max() - pd.DateOffset(years=VALID_YEARS)
    + pd.Timedelta(days=1)
)

train_fit = train.loc[
    :validation_start - pd.Timedelta(days=H + 1)
].copy()

validation = train.loc[
    validation_start:
].copy()

print(
    f"Model fitting: {train_fit.index.min().date()} → "
    f"{train_fit.index.max().date()} | n={len(train_fit)}"
)

print(
    f"Validation:    {validation.index.min().date()} → "
    f"{validation.index.max().date()} | n={len(validation)}"
)

assert (
    train_fit.index.max() + pd.Timedelta(days=H)
    < validation.index.min()
), "Validation embargo violated"

In [ ]:
# Define feature groups to evaluate the contribution of additional weather and hydrological variables
# Compare different feature combinations to identify which inputs improve water-level forecasting

feature_groups = {
    "Kaub only": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28"
    ],

    "Kaub + precipitation": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28",
        "precip_3d", "precip_7d",
        "precip_14d", "precip_30d"
    ],

    "Kaub + precipitation + temperature": [
        "lag_0", "lag_1", "lag_2", "lag_3",
        "lag_7", "lag_14", "lag_28",
        "precip_3d", "precip_7d",
        "precip_14d", "precip_30d",
        "temp_3d", "temp_7d"
    ],

     #"Kaub + weather + Bingen + discharge": [
      #  "lag_0", "lag_1", "lag_2", "lag_3",
       # "lag_7", "lag_14", "lag_28",

        # Precipitation
        #"precip_3d", "precip_7d",
        #"precip_14d", "precip_30d",

        # Temperature
        #"temp_3d", "temp_7d",

        # Bingen water level
        #"bingen_lag_1", "bingen_lag_2",
        #"bingen_lag_3", "bingen_lag_7",

        # Kaub discharge
        #"discharge_lag_1", "discharge_lag_3",
        #"discharge_lag_7", "discharge_7d_mean"
    #],

    "All selected features": FEATURES
}

In [ ]:
# Train and evaluate Linear Regression models using different feature groups
# Compare validation MAE and RMSE to assess the impact of additional features on forecasting accuracy

import pandas as pd #
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []

for group_name, columns in feature_groups.items():

    model = LinearRegression()

    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    predictions = model.predict(validation[columns])

    mae = mean_absolute_error(
        validation["target"],
        predictions
    )

    rmse = mean_squared_error(
        validation["target"],
        predictions
    ) ** 0.5

    results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse
    })

results_df = pd.DataFrame(results)

display(results_df.round(2))

In [ ]:
# Evaluate the persistence baseline by assuming the water level remains unchanged after 7 days
# Calculate MAE and RMSE to compare baseline performance with the machine learning models

persistence_predictions = validation["lag_0"]

persistence_mae = mean_absolute_error(
    validation["target"],
    persistence_predictions
)

persistence_rmse = mean_squared_error(
    validation["target"],
    persistence_predictions
) ** 0.5

print(f"Persistence MAE:  {persistence_mae:.2f} cm")

print(f"Persistence RMSE: {persistence_rmse:.2f} cm")

In [ ]:
# Calculate the relative improvement in MAE compared with the persistence baseline
# Positive skill indicates better forecasting performance than persistence

results_df["Skill vs persistence"] = (
    1 - results_df["MAE"] / persistence_mae
)

display(results_df.round(3))

1. Modeling : LightGBM

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

In [ ]:
# Store results and trained models
lgbm_results = []
lgbm_models = {}

# Loop through each feature group
for group_name, columns in feature_groups.items():

    # Initialize LightGBM model
    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.2,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    # Train model
    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    # Predict on validation data
    predictions = model.predict(
        validation[columns]
    )

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        validation["target"],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            validation["target"],
            predictions
        )
    )

    R2 = r2_score(
        validation["target"],
        predictions
    )

    # Calculate skill vs persistence
    skill = 1 - mae / persistence_mae

    # Store results
    lgbm_results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse,
        "R_squared": R2,
        "Skill vs persistence": skill
    })

    # Store trained model
    lgbm_models[group_name] = model

# Create results DataFrame
lgbm_results_df = pd.DataFrame(lgbm_results)

# Display results
display(lgbm_results_df.round(3))

2. Modeling : XGBoost

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import pandas as pd
import numpy as np

# Train and evaluate XGBoost models using different feature groups
# Compare MAE, RMSE, R², Bias, and skill against the persistence baseline

xgb_results = []
xgb_models = {}

for group_name, columns in feature_groups.items():

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        train_fit[columns],
        train_fit["target"]
    )

    predictions = model.predict(validation[columns])

    mae = mean_absolute_error(
        validation["target"],
        predictions
    )

    rmse = mean_squared_error(
        validation["target"],
        predictions
    ) ** 0.5

    R2 = r2_score(
        validation["target"],
        predictions
    )

    bias = np.mean(
        predictions - validation["target"]
    )

    xgb_results.append({
        "Feature group": group_name,
        "Number of features": len(columns),
        "MAE": mae,
        "RMSE": rmse,
        "R_square": R2,
        "Bias": bias,
        "Skill vs persistence": 1 - mae / persistence_mae
    })

    xgb_models[group_name] = model

xgb_results_df = pd.DataFrame(xgb_results)

display(xgb_results_df.round(3))

In [ ]:
linear_comparison = results_df.copy()
linear_comparison["Model"] = "Linear Regression"

xgb_comparison = xgb_results_df.copy()
xgb_comparison["Model"] = "XGBoost"

comparison = pd.concat(
    [linear_comparison, xgb_comparison],
    ignore_index=True
)

comparison = comparison[
    [
        "Model",
        "Feature group",
        "Number of features",
        "MAE",
        "RMSE",
        "Skill vs persistence"
    ]
]

display(comparison.round(3))

In [ ]:

model = xgb_models["All selected features"]

importance = pd.Series(
    model.feature_importances_,
    index=feature_groups["All selected features"]
).sort_values()

importance.plot(
    kind="barh",
    figsize=(9, 7),
    title="XGBoost feature importance — Day+7 forecast"
)

plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

Testing :

In [ ]:
best_group = xgb_results_df.loc[
    xgb_results_df["MAE"].idxmin(),
    "Feature group"
]

FEATURES = feature_groups[best_group]

print("Selected feature group:", best_group)
print("Number of features:", len(FEATURES))
print("Features:", FEATURES)

In [ ]:
from xgboost import XGBRegressor
import pandas as pd

# Combine training and validation data
train_final = pd.concat(
    [train_fit, validation],
    axis=0
).sort_index()

# Ensure the test period comes after training and validation
assert train_final.index.max() < test.index.min(), \
    "Training and test periods overlap!"

# Separate features and target
X_train_final = train_final[FEATURES]
y_train_final = train_final["target"]

X_test = test[FEATURES]
y_test = test["target"]

# Initialize the final model with the same hyperparameters
final_xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Train the final model
final_xgb_model.fit(X_train_final, y_train_final)

# Predict on the untouched test data
test_predictions = final_xgb_model.predict(X_test)

In [ ]:
import pandas as pd

pd.DataFrame(test_predictions, columns=["Predicted Water Level"])

In [ ]:
test_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": test_predictions
})

display(test_results)

In [ ]:
#Compare actual and predicted values

test_results = pd.DataFrame({
    "Actual": np.asarray(y_test).ravel(),
    "Predicted": np.asarray(test_predictions).ravel()
}, index=y_test.index)

test_results["Absolute_Error"] = abs(
    test_results["Actual"] - test_results["Predicted"]
)

display(test_results.round(2))

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    test_results["Actual"],
    test_results["Predicted"]
)

rmse = np.sqrt(
    mean_squared_error(
        test_results["Actual"],
        test_results["Predicted"]
    )
)

r2 = r2_score(
    test_results["Actual"],
    test_results["Predicted"]
)

print(f"MAE  : {mae:.2f} cm")
print(f"RMSE : {rmse:.2f} cm")
print(f"R²   : {r2:.3f}")

In [ ]:
#Plot comparison

plt.figure(figsize=(14, 5))

plt.plot(
    test_results.index,
    test_results["Actual"],
    label="Actual",
    alpha=0.8
)

plt.plot(
    test_results.index,
    test_results["Predicted"],
    label="Predicted",
    alpha=0.8
)

plt.title("Actual vs Predicted Water Levels at Kaub")
plt.xlabel("Date")
plt.ylabel("Water Level (cm)")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

Real-time Data analysis

In [ ]:
import requests
import pandas as pd

# PEGELONLINE API
BASE_URL = "https://www.pegelonline.wsv.de/webservices/rest-api/v2"

STATION_ID = "1d26e504-7f9e-480a-b52c-5932be6549ab"

# Fetch the last 30 days of measurements
url = (
    f"{BASE_URL}/stations/{STATION_ID}"
    "/W/measurements.json"
)

response = requests.get(
    url,
    params={"start": "P30D"},
    timeout=30
)

response.raise_for_status()

# Convert API response into DataFrame
df_live = pd.DataFrame(response.json())

# Convert timestamp
df_live["timestamp"] = pd.to_datetime(
    df_live["timestamp"],
    utc=True
).dt.tz_convert("Europe/Berlin")

# Rename column
df_live = df_live.rename(
    columns={"value": "water_level"}
)

# Display latest measurements
display(df_live.tail(10))

In [ ]:
# Extract local calendar date
df_live["date"] = df_live["timestamp"].dt.date

# Aggregate measurements to daily values
df_daily_live = (
    df_live.groupby("date")
    .agg(
        water_level_mean=("water_level", "mean"),
        water_level_min=("water_level", "min"),
        water_level_max=("water_level", "max")
    )
    .reset_index()
)

df_daily_live["date"] = pd.to_datetime(
    df_daily_live["date"]
)

df_daily_live = df_daily_live.set_index("date")

display(df_daily_live.tail(10).round(2))

In [ ]:
# Extract date
df_live["date"] = df_live["timestamp"].dt.date

# Aggregate measurements to daily values
df_daily_live = (
    df_live.groupby("date")
    .agg(
        water_level_mean=("water_level", "mean"),
        water_level_min=("water_level", "min"),
        water_level_max=("water_level", "max")
    )
    .reset_index()
)

# Convert date to datetime
df_daily_live["date"] = pd.to_datetime(
    df_daily_live["date"]
)

# Set date as index
df_daily_live = df_daily_live.set_index("date")

# Display last 10 days
display(df_daily_live.tail(10).round(2))

In [ ]:
print("Total days:", len(df_daily_live))

print("Latest available date:", df_daily_live.index.max())

print("Missing values:")
print(df_daily_live.isna().sum())

In [ ]:
# Check the number of measurements per day
daily_counts = (
    df_live.groupby("date")
    .size()
    .rename("measurements")
)

display(daily_counts.tail(5))

In [ ]:
# Use only completed calendar days
today = pd.Timestamp.now(
    tz="Europe/Berlin"
).date()

df_daily_complete = df_daily_live.loc[
    df_daily_live.index.date < today
].copy()

display(df_daily_complete.tail())

In [ ]:
# Copy the completed daily data
df_features = df_daily_complete.copy()

# Create Kaub lag features
df_features["kaub_lag_1"] = (
    df_features["water_level_mean"].shift(1)
)

df_features["kaub_lag_2"] = (
    df_features["water_level_mean"].shift(2)
)

df_features["kaub_lag_3"] = (
    df_features["water_level_mean"].shift(3)
)

df_features["kaub_lag_7"] = (
    df_features["water_level_mean"].shift(7)
)

df_features["kaub_lag_14"] = (
    df_features["water_level_mean"].shift(14)
)

# Water-level change
df_features["kaub_change_1d"] = (
    df_features["water_level_mean"]
    - df_features["water_level_mean"].shift(1)
)

display(df_features.tail())

In [ ]:
import requests
import pandas as pd

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 50.08,
    "longitude": 7.76,
    "start_date": "2026-09-21",
    "end_date": "2026-09-21",
    "daily": [
        "precipitation_sum",
        "temperature_2m_mean"
    ],
    "timezone": "Europe/Berlin"
}

response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

weather_data = response.json()

df_missing_weather = pd.DataFrame(
    weather_data["daily"]
)

df_missing_weather = df_missing_weather.rename(
    columns={
        "time": "date",
        "temperature_2m_mean": "temperature_mean"
    }
)

df_missing_weather["date"] = pd.to_datetime(
    df_missing_weather["date"]
)

df_missing_weather = df_missing_weather.set_index("date")

display(df_missing_weather)

In [ ]:
df_realtime = df_daily_complete.join(
    df_weather_live,
    how="left"
)

display(df_realtime.tail(10))

In [ ]:
selected_group = "Kaub + precipitation + temperature"

# Retrieve the trained XGBoost model
xgb_model = xgb_models[selected_group]

# Retrieve the exact feature names
FEATURES = list(xgb_model.feature_names_in_)

print("Number of features:", len(FEATURES))

for feature in FEATURES:
    print(feature)

In [ ]:
selected_group = "Kaub + precipitation + temperature"

# Retrieve the trained model
xgb_model = xgb_models[selected_group]

# Get the exact features used during training
FEATURES = list(xgb_model.feature_names_in_)

print("Model features:")
print(FEATURES)

In [ ]:
# Copy your merged real-time dataset
df_live_features = df_realtime.copy()

# Sort chronologically
df_live_features = df_live_features.sort_index()

# Check for missing values
print(df_live_features.isna().sum())

# Display latest observations
display(df_live_features.tail())

In [ ]:
# Kaub water-level lag features
for lag in [1, 2, 3, 7, 14]:
    df_live_features[f"kaub_lag_{lag}"] = (
        df_live_features["water_level_mean"].shift(lag)
    )

# Daily water-level change
df_live_features["kaub_change_1d"] = (
    df_live_features["water_level_mean"]
    - df_live_features["water_level_mean"].shift(1)
)

# Rolling precipitation
for window in [3, 7, 14, 30]:
    df_live_features[f"precip_{window}d"] = (
        df_live_features["precipitation_sum"]
        .rolling(window=window)
        .sum()
    )

# Rolling temperature
for window in [3, 7]:
    df_live_features[f"temp_{window}d_mean"] = (
        df_live_features["temperature_mean"]
        .rolling(window=window)
        .mean()
    )

display(df_live_features.tail())

In [ ]:
print("Total rows:", len(df_realtime))

print("First date:", df_realtime.index.min())
print("Last date:", df_realtime.index.max())

print("\nMissing values:")
print(
    df_realtime[
        ["precipitation_sum", "temperature_mean"]
    ].isna().sum()
)

print("\nLast 5 days:")
display(df_realtime.tail())

In [ ]:
weather_columns = [
    "precipitation_sum",
    "temperature_mean"
]

# Update missing weather values
df_realtime[weather_columns] = (
    df_realtime[weather_columns].combine_first(
        df_missing_weather[weather_columns]
    )
)

display(df_realtime.tail())

In [ ]:
# Copy and sort data
df_live_features = df_realtime.sort_index().copy()

# Kaub lag features
for lag in [1, 2, 3, 7, 14]:
    df_live_features[f"kaub_lag_{lag}"] = (
        df_live_features["water_level_mean"].shift(lag)
    )

# Daily water-level change
df_live_features["kaub_change_1d"] = (
    df_live_features["water_level_mean"]
    - df_live_features["water_level_mean"].shift(1)
)

# Rolling precipitation
for window in [3, 7, 14, 30]:
    df_live_features[f"precip_{window}d"] = (
        df_live_features["precipitation_sum"]
        .rolling(window=window)
        .sum()
    )

# Rolling temperature
for window in [3, 7]:
    df_live_features[f"temp_{window}d_mean"] = (
        df_live_features["temperature_mean"]
        .rolling(window=window)
        .mean()
    )

display(df_live_features.tail())

In [ ]:
# Copy your real-time dataset
df_live_features = df_realtime.sort_index().copy()

# Kaub water-level lag features
for lag in [0, 1, 2, 3, 7, 14, 28]:

    df_live_features[f"lag_{lag}"] = (
        df_live_features["water_level_mean"].shift(lag)
    )

# Temperature features
df_live_features["temp_3d"] = (
    df_live_features["temperature_mean"]
    .rolling(window=3)
    .mean()
)

df_live_features["temp_7d"] = (
    df_live_features["temperature_mean"]
    .rolling(window=7)
    .mean()
)

display(df_live_features.tail())

In [ ]:
# Get the exact model features
FEATURES = list(xgb_model.feature_names_in_)

print("Required model features:")

for feature in FEATURES:
    print(feature)

In [ ]:
# Create rolling precipitation features

for window in [3, 7, 14, 30]:

    df_live_features[f"precip_{window}d"] = (
        df_live_features["precipitation_sum"]
        .rolling(window=window)
        .sum()
    )

# Display the latest observations
display(df_live_features.tail())

In [ ]:
# Retrieve exact model features
FEATURES = list(xgb_model.feature_names_in_)

# Check whether all required columns exist
missing_columns = [
    col for col in FEATURES
    if col not in df_live_features.columns
]

print("Missing columns:", missing_columns)

if missing_columns:
    raise ValueError(
        f"Missing features: {missing_columns}"
    )

# Select the latest observation
latest_date = df_live_features.index.max()

X_live = df_live_features.loc[
    [latest_date], FEATURES
].copy()

display(X_live)

In [ ]:
print("Missing values in model input:")

print(X_live.isna().sum())

In [ ]:
# Validate input
if X_live.isna().any().any():
    raise ValueError(
        "Missing values detected. "
        "Complete the required input data first."
    )

# Generate 7-day-ahead prediction
prediction = xgb_model.predict(X_live)

predicted_level = float(prediction[0])

forecast_date = (
    latest_date + pd.Timedelta(days=7)
)

print("Station: Kaub")
print("Model: XGBoost")

print(f"Latest observation: {latest_date.date()}")
print(f"Forecast date: {forecast_date.date()}")

print(
    f"Predicted water level: {predicted_level:.2f} cm"
)